# 11 - Crypto Sector Rotation & Real-Time Alert Engine

Sector momentum aggregation, market breadth calculations, rotation quadrant classification (Leading, Accelerating, Weakening, Declining), and automated signal alert broadcasting.


In [ ]:
import asyncio

from src.alerts.engine import AlertEngine
from src.alerts.models import AlertSeverity, AlertType
from src.database.session import get_session_factory, init_db
from src.features.sector import get_all_sector_performances

print('Sector rotation and alert modules loaded.')


In [ ]:
async def analyze_sectors_and_alerts():
    await init_db()
    factory = get_session_factory()
    async with factory() as session:
        sectors = await get_all_sector_performances(session)
        print('=== SECTOR ROTATION MATRIX ===')
        for s in sectors:
            print(f'{s.sector_name:<16} | 7D: {s.return_7d*100:+.1f}% | Breadth: {s.breadth_pct:4.1f}% | {s.rotation_status}')

    # Test alert dispatching
    alert_engine = AlertEngine(cooldown_minutes=15)
    alert = await alert_engine.emit(
        alert_type=AlertType.MOMENTUM_BREAKOUT,
        severity=AlertSeverity.INFO,
        symbol='SOL',
        message='SOL 24h momentum breakout triggered from sector rotation leader.',
    )
    if alert:
        print(f'\nDispatched alert: [{alert.severity.value}] {alert.message}')

asyncio.run(analyze_sectors_and_alerts())
